# Step 21.4 — Top-k：筛选原 ID，保留词表轴，再归一化

在仓库根目录先同步母版：

```bash
git pull
```

再建立独立工作副本，不覆盖已有笔记（仓库已有 notebooks/work 目录）：

```bash
cp -n notebooks/95_step21_4_top_k_sampling.ipynb notebooks/work/95_step21_4_top_k_sampling_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上命令仅供未来使用，本次未执行。沿用项目 Python 3.12 / .venv，API 固定 **MLX v0.32.2**。

2026-11-17 教学修订：**21.4 累计 1/2＝历史修正 0/2＋本次教学修订 1 次**。两段原代码和练习原样保留；只补纸算、源码解读与相邻接口，不实现新 chooser 或生产 kernel。

所有代码单元 execution_count=null、outputs=[]。本次不执行 Notebook、MLX、测试、采样、训练、实验、语料或 checkpoint 操作；纸算不是运行结果，工作副本也未被操作。


## 1. 从温度到候选集合：增加一个不同的选择条件

[21.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/9b08bb9a300618cb42f7e63b20314ef873001b9e/notebooks/94_step21_3_temperature.ipynb)通过正温度改变相对概率；本节再用 **top-k** 限制候选集合，只保留最高分的 k 个原词表 ID。其余候选在数学分布中概率为 0。这叫改变 **support（支持集）**，与只缩放 logits 不同。

本函数采用“严格 k 个不同索引”：分数相同时用 ID 升序打破并列。它不采用“分数不小于第 k 大值”的阈值法；两种政策在并列处可能保留不同数量，本节只实现前者。

保留哪些 ID、怎样表示屏蔽、怎样抽一个 ID，是三个步骤。top-k helper 只完成前两步，不调用模型、不采样，也不改变模型权重。


## 2. 读懂排序键与负无穷 mask

输入 `logits[i]` 是原 ID=i 的分数。代码排序的是 `range(V)` 这列 **ID**，不是原地重排 logits：

```text
sorted(range(V), key=lambda i: (-values[i], i))[:k]
```

sorted 默认升序。二元 tuple 先比较第一项，相同时才比较第二项：把 score 取负，使高分更靠前；同分时第二项 i 较小者更靠前。lambda 是这里临时使用的“从 ID 算出比较键”的小函数；[:k] 取排序结果前 k 项。[Python 排序规则](https://docs.python.org/3.12/howto/sorting.html)说明 key 函数与这种逐项比较。

得到保留集合 S 后，**masked logits（屏蔽后的分数）**仍有完整 V 个位置：

$$
z'_i=\begin{cases}z_i&i\in S\\-\infty&i\notin S,\end{cases}
\qquad
p'_i=\begin{cases}
\dfrac{\exp(z_i)}{\sum_{j\in S}\exp(z_j)}&i\in S\\
0&i\notin S.
\end{cases}
$$

因为 exp(−∞)=0，屏蔽项不进入归一化分母。用 0 代替 −∞ 不等价：exp(0)=1，仍给被排除项留下权重。categorical 接收 z'，无需先把 softmax 概率传进去。


## 3. 四元素纸算：排序后列表不等于新词表

取人工分数 `z=[ln 3,0,ln 6,ln 3]`，V=4、k=2。指数权重为 [3,1,6,3]，未筛选时总和是 13。下面逐个**原 ID**保留行位置：

| 原 ID | score | 排序键 (−score,ID) | 是否保留 | masked logit | 重新归一化概率 |
|---:|---|---|---|---|---|
| 0 | ln 3 | (−ln 3,0) | 是 | ln 3 | 3/9=1/3 |
| 1 | 0 | (0,1) | 否 | −∞ | 0 |
| 2 | ln 6 | (−ln 6,2) | 是 | ln 6 | 6/9=2/3 |
| 3 | ln 3 | (−ln 3,3) | 否 | −∞ | 0 |

ID 2 的分数最高；ID 0 和 3 同分，比较键第二项后 0 在 3 前。完整排序 ID 列表是 [2,0,3,1]，所以 `selected=[2,0]`，支持集 S={0,2}。

对应布尔 mask 是 [True,False,True,False]，masked logits 是 [ln 3,−∞,ln 6,−∞]，**长度仍为 4**。不是把它缩成 [ln 6,ln 3] 后重新把两个位置编号为 0、1。

保留项原概率分别为 3/13 与 6/13，总和只有 9/13。直接删除其余概率还不够，需除以剩余总和，得到 [1/3,0,2/3,0]。本表是数学推导，不是实际采样或 float32 打印结果。

若未来从这条完整 V 轴的分布抽到 ID 2，它已经是原词表的 ID 2；不能再拿它去索引 selected。`selected` 用于说明保留了谁，不是这个返回 ID 还需经过的一张映射表。


### 3.1 同一个例子，阈值法为什么留下三个？

上例第 2 大分数是 ln 3。若改用 score≥ln 3，则 ID 0、2、3 全部通过，共 3 个；这与本函数 selected=[2,0] 的 2 个不同。

本函数通过排序后的前 k 个索引定义并列政策，而不是事后找一个分数阈值。两种定义都可以单独采用，但不能用阈值法的结果解释这份代码。此处也没有根据累计概率选择集合，不是 top-p/nucleus sampling。


## 4. 将纸算逐段对应到实际 helper

| 源码阶段 | 输入／输出与作用 |
|---|---|
| ndim 与长度检查 | 只接受非空单行 (V,)；不是 (B,V) 接口 |
| k 检查 | Python int 且 1≤k≤V；bool 也不被接受 |
| astype(float32)、eval、tolist | 得到已求值分数的 host Python list，长度 V |
| isfinite 检查 | 拒绝转换后的 NaN、+Inf、−Inf 输入 |
| sorted(... )[:k] | 得到 k 个原 ID 的 Python list，按 score/tie policy 排列 |
| [i in selected for i in range(V)] | 按原 ID 顺序生成 (V,) bool mask |
| mx.where(keep,z,−Inf) | 真位置保留分数，假位置屏蔽；输出仍为 (V,) |
| return | 返回 **(filtered_logits, selected_IDs)**，不是 token 或概率 |

下面代码已足够短，原样保留，不为行数拆出新 helper。注意排序列表与最终输出数组使用的顺序不同：列表按分数排，数组的轴仍按原 token ID 排。


In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def exact_top_k_logits(logits, k):
    if logits.ndim != 1 or logits.shape[0] == 0:
        raise ValueError("expect one nonempty vocabulary row (V,)")
    V = logits.shape[0]
    if type(k) is not int or not 1 <= k <= V:
        raise ValueError("k must be an integer in [1,V]")
    z = logits.astype(mx.float32)
    mx.eval(z)
    values = z.tolist()
    if any(not math.isfinite(x) for x in values):
        raise ValueError("input logits must be finite before filtering")
    selected = sorted(range(V), key=lambda i: (-values[i], i))[:k]
    keep = mx.array([i in selected for i in range(V)], dtype=mx.bool_)
    return mx.where(keep, z, float("-inf")), selected

### 4.1 为什么这是教学参考，不是生产 top-k kernel？

mx.eval(z) 建立求值边界，tolist() 把 V 个分数转成 host 上的 Python 数值；之后 sorted 与列表判断都在 Python 层进行。Apple Silicon 的统一内存并不让 Python 对象构造或这些循环免费。

从算法结构估算：全量排序通常按 O(V log V) 的比较成本考虑；O 记号在这里表示随输入大小增长的工作量级，不是秒数。本代码的 mask 还使用 **list** 成员判断，每个 i 最坏扫描 k 个 selected 元素，共 O(Vk)。另外仍要建立长度 V 的 mask 与 filtered logits，不能因为只选 k 项就把所有后续数组都当成长度 k。

这些是源码层面的成本说明，不是实测速度、峰值内存或 benchmark。本节保留透明的 tie policy 和原 ID 关系，不把它包装成 fused kernel，也不在这轮改造为另一套实现。


## 5. 未来观察：selected 的顺序会改变概率数组的位置吗？

先用四元素表回答这个问题，再决定是否在未来工作副本观察。下一个单元保留原来的**三候选**例子 [ln 6,ln 3,0]，用来查看 selected、完整概率向量与一次抽样；它不是上面四元素并列例子的运行记录。本轮不为统一例子而改写这段有效源码。

未来可检查三件不同的事：filtered/p 是否保留原 V 长度；被屏蔽位置和归一化是否符合数学定义；一次抽到的 token 是否属于保留集合。一次抽样不能验证概率频率或模型质量，也不意味着每个保留 ID 都应该出现。

这里直接以显式 key(23) 做一次 categorical，返回的是单行 (V,) 输入对应的标量 ID，所以 token.item() 合法。当前没有运行，seed=23 会选哪项不在本页伪填。接入连续生成时则按第 6 节沿用 21.2 的 key 传递。


In [ ]:
z = mx.array([math.log(6), math.log(3), 0.], dtype=mx.float32)
filtered, selected = exact_top_k_logits(z, 2)
p = mx.softmax(filtered, axis=-1)
token = mx.random.categorical(filtered, key=mx.random.key(23))
mx.eval(p, token)
print("kept original IDs:", selected, "p:", p.tolist(), "draw:", token.item())

## 6. 实际衔接：温度在前，屏蔽在后

[21.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/9b08bb9a300618cb42f7e63b20314ef873001b9e/notebooks/94_step21_3_temperature.ipynb)的 temperature_logits 接受有限 logits，[21.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/9b08bb9a300618cb42f7e63b20314ef873001b9e/notebooks/93_step21_2_greedy_vs_sampling.ipynb)的 choose_sample 接受 logits 与显式 key 并返回 (token,next_key)。本节 exact_top_k_logits 只处理一个非空 (V,) 行。

对现有 B=1 的生成循环，形状应逐步对应如下；这只是已有接口的连接说明，没有新增运行入口或改造 chooser：

| 步骤 | 实际接口／操作 | 输出 |
|---|---|---|
| 从模型当前前缀取末行 | logits[:, -1, :] | last (1,V) |
| 温度缩放 | temperature_logits(last,τ) | scaled (1,V) |
| 取出一行给 top-k | scaled[0] | (V,) |
| 筛选 | filtered,selected = exact_top_k_logits(scaled[0],k) | (V,) 与长度 k 的原 ID 列表 |
| 恢复 batch 轴 | filtered[None,:] | (1,V) |
| 按筛选后的分数抽样 | token,next_key = choose_sample(filtered[None,:],key) | token (1,) int32 与 next_key |
| 准备 append | token[:,None]，然后沿 axis=1 concatenate | 新一列 (1,1)，保留原 history |

下一轮保留新 ids **和 next_key**，不能重复使用旧 key。只把 token 加入序列，不拼接整个返回 tuple，也不把 selected 当作 token。若直接把 (V,) 传给 choose_sample，token 会是 shape ()；不能随后照抄 token[:,None]，所以表中先恢复 (1,V)。

温度函数与本节 helper 都要求输入有限；top-k 之后出现的 −Inf 可以送给 categorical，却不能再送回它们的有限性检查。正确顺序是先温度、后屏蔽、再选择，不能先屏蔽再温度。

一般 (B,V) 可按行考虑，但本 top-k helper 不是 batch 实现；上表只连接已有单条 prompt 的 B=1 路径，不据此声称完整批量、EOS 或 KV-cache 生成能力。


### 6.1 k、mask 与浮点边界

- k=1 保留一个按本 tie policy 选出的索引，数学上只剩该候选；若仍调用 choose_sample，仍会拆分并返回 key，并不自动切换为 greedy 分支。k=V 不屏蔽任何位置。函数拒绝 k=0，避免全行 −Inf 没有合法分布。
- 输入先转成 float32，再检查、排序。转换造成的非有限值会被拒绝；原本很接近的不同分数若舍入成相等，就会按实际 float32 值进入 tie-break。不能把结果当作对任意更高精度原数值的排序保证。
- 正温度在精确算术下保持分数顺序；实际极端缩放可能丢失差异，因此不能无条件承诺数值实现的 top-k 集合不变。有限性检查也不等于排序误差检查。
- “严格 k”指保留 k 个有限 logit 位置。数学上它们都有正概率，有限精度的极小概率仍可能下溢，有限次采样也未必见到全部 k 项；不要把 k 解释为一定观察到 k 种输出。

这里的 −Inf 是明确排除候选的 mask，而不是概率零、随机 key 或索引。helper 不负责验证采样器的整套使用流程，仍需保持正确的 V、ID 映射和上下文约定。


## 7. 练习（不附答案）

1. 对四个不同分数手算 k=2 的归一化概率。
2. 设计包含三个并列最大值的例子，对比严格 k 与阈值法。
3. 解释为何返回排序后位置而不是原 token ID 会导致错误解码。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 固定版本依据与材料状态

- [Python 3.12 sorting](https://docs.python.org/3.12/howto/sorting.html)：key 函数、tuple 按项比较、sorted 返回新列表。
- [MLX v0.32.2 array.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/array.cpp)：tolist 的 Python 表示与 astype。
- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：where 按布尔条件保留／替换元素、softmax。
- [MLX v0.32.2 random.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp)：categorical 的 logits、axis、输出 shape 与 key。

本次只修订讲解：四元素排序／tie-break／mask／归一化纸算、host 成本及相邻接口。原代码和练习未变，未运行课程代码、采样、测试、绘图、训练或实验，未读取语料／checkpoint；没有真实输出或性能结果。

严格 JSON/schema、唯一合法 ID、语法与 AST 核验属于静态检查。21.4 累计 **1/2**（历史 0/2＋本次教学修订 1 次），其他历史计数保持；所有代码单元未执行、输出为空。本轮停在 21.4，等待独立验收。
